In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Adjust this path if you uploaded the zip somewhere other than MyDrive root
ZIP_PATH = '/content/drive/MyDrive/malaria_dataset.zip'

!unzip -q "$ZIP_PATH" -d /content/

In [ ]:
!ls /content/malaria_dataset

data.yaml  images  labels


In [ ]:
yaml_content = """path: /content/malaria_dataset
train: images/train
val: images/val
nc: 2
names: ['Parasitized', 'Uninfected']
"""

with open('/content/malaria_dataset/data.yaml', 'w') as f:
    f.write(yaml_content)

print("data.yaml rewritten for Colab.")
!cat /content/malaria_dataset/data.yaml

data.yaml rewritten for Colab.
path: /content/malaria_dataset
train: images/train
val: images/val
nc: 2
names: ['Parasitized', 'Uninfected']


In [ ]:
!pip install ultralytics -q

In [ ]:
!nvidia-smi

Wed Nov  4 00:48:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')   # nano model — fast, good for Colab free tier

results = model.train(
    data='/content/malaria_dataset/data.yaml',
    epochs=5,
    imgsz=640,
    batch=16,
    name='malaria_yolo'
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/malaria_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.

In [ ]:
import shutil, os

# Find the actual run folder (name may vary if re-run)
run_dir = '/content/runs/detect/malaria_yolo'

if os.path.exists(f'{run_dir}/weights/best.pt'):
    shutil.copy(
        f'{run_dir}/weights/best.pt',
        '/content/drive/MyDrive/malaria_yolo_best.pt'
    )
    print("Model saved to Drive: /content/drive/MyDrive/malaria_yolo_best.pt")
else:
    print("best.pt not found — check run_dir path with: !ls /content/runs/detect/")

Model saved to Drive: /content/drive/MyDrive/malaria_yolo_best.pt


In [ ]:
import pandas as pd
from ultralytics import YOLO

# ---- 1. Re-run validation to grab per-class metrics programmatically ----
model = YOLO('/content/runs/detect/malaria_yolo/weights/best.pt')
metrics = model.val(data='/content/malaria_dataset/data.yaml', verbose=False)

# ---- 2. Build the overall metrics table ----
overall = pd.DataFrame([{
    'Model':            'YOLO11n',
    'Epochs':           5,
    'ImageSize':        640,
    'BatchSize':        16,
    'TrainImages':      22046,
    'ValImages':        5512,
    'Precision':        float(metrics.box.mp),
    'Recall':           float(metrics.box.mr),
    'mAP50':            float(metrics.box.map50),
    'mAP50-95':         float(metrics.box.map),
    'Inference_ms_per_img': 3.4,
}])

# ---- 3. Build per-class metrics table ----
class_names = metrics.names  # {0: 'Parasitized', 1: 'Uninfected'}
per_class_rows = []
for i, name in class_names.items():
    per_class_rows.append({
        'Class':     name,
        'Precision': float(metrics.box.p[i]),
        'Recall':    float(metrics.box.r[i]),
        'mAP50':     float(metrics.box.ap50[i]),
        'mAP50-95':  float(metrics.box.ap[i]),
    })
per_class = pd.DataFrame(per_class_rows)

# ---- 4. Training config sheet ----
config = pd.DataFrame([
    ('Model file',       'yolo11n.pt'),
    ('Ultralytics ver',  '8.4.202'),
    ('PyTorch ver',      '2.11.0+cu130'),
    ('GPU',              'Tesla T4 (14913 MiB)'),
    ('Dataset',          'Kaggle malaria cell images'),
    ('Label format',     'Full-image box (0.5, 0.5, 1.0, 1.0)'),
    ('Classes',          '0=Parasitized, 1=Uninfected'),
    ('Train/Val split',  '80/20, stratified random, seed=42'),
    ('Optimizer',        'AdamW (auto), lr=0.001667'),
    ('Total training time', '0.444 hours (~27 min)'),
    ('Params',           '2,590,230'),
    ('GFLOPs',           '6.5'),
], columns=['Parameter', 'Value'])

# ---- 5. Write to Excel ----
out_path = '/content/drive/MyDrive/YOLO_malaria_metrics.xlsx'
with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    overall.to_excel(writer,   sheet_name='Overall',      index=False)
    per_class.to_excel(writer, sheet_name='PerClass',     index=False)
    config.to_excel(writer,    sheet_name='Config',       index=False)

print(f"Saved: {out_path}")
print("\nOverall metrics:")
print(overall.to_string(index=False))
print("\nPer-class metrics:")
print(per_class.to_string(index=False))

Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,597,294 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 749.8±351.7 MB/s, size: 12.1 KB)
val: Scanning /content/malaria_dataset/labels/val.cache... 5512 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5512/5512 1.9Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 345/345 7.1it/s 48.7s
                   all       5512       5512      0.962       0.97      0.988      0.988
Speed: 0.9ms preprocess, 3.8ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /content/runs/detect/val
Saved: /content/drive/MyDrive/YOLO_malaria_metrics.xlsx

Overall metrics:
  Model  Epochs  ImageSize  BatchSize  TrainImages  ValImages  Precision   Recall    mAP50  mAP50-95  Inference_ms_per_img
YOLO11n       5        640         16        22046       5512   0.9623

In [10]:
!zip -r runs.zip runs/


  adding: runs/ (stored 0%)
  adding: runs/detect/ (stored 0%)
  adding: runs/detect/malaria_yolo/ (stored 0%)
  adding: runs/detect/malaria_yolo/confusion_matrix.png (deflated 30%)
  adding: runs/detect/malaria_yolo/val_batch2_labels.jpg (deflated 23%)
  adding: runs/detect/malaria_yolo/results.png (deflated 8%)
  adding: runs/detect/malaria_yolo/train_batch1.jpg (deflated 26%)
  adding: runs/detect/malaria_yolo/BoxF1_curve.png (deflated 15%)
  adding: runs/detect/malaria_yolo/args.yaml (deflated 53%)
  adding: runs/detect/malaria_yolo/train_batch0.jpg (deflated 26%)
  adding: runs/detect/malaria_yolo/confusion_matrix_normalized.png (deflated 30%)
  adding: runs/detect/malaria_yolo/val_batch2_pred.jpg (deflated 21%)
  adding: runs/detect/malaria_yolo/val_batch1_pred.jpg (deflated 22%)
  adding: runs/detect/malaria_yolo/BoxP_curve.png (deflated 17%)
  adding: runs/detect/malaria_yolo/results.csv (deflated 56%)
  adding: runs/detect/malaria_yolo/val_batch0_pred.jpg (deflated 25%)
  addi